# Customer Lifetime Value Analysis

> **Goal:** estimate and compare customer lifetime value (CLV) across customer segments,
> channel behavior, and retention patterns using the marketing campaign customer base.
> The notebook combines an industry-standard BG/NBD + Gamma-Gamma workflow with
> business-friendly comparisons and driver analysis.

| Dimension | Business question |
|---|---|
| CLV level | Which customers are worth the most over the next 12 months? |
| Segments | Which income, family, and education groups create the most value? |
| Channels | Which dominant purchase channels are associated with higher CLV? |
| Retention | How does probability-alive translate into future value? |
| Drivers | What variables most strongly explain CLV differences? |

> **Important note:** the dataset does not include the original acquisition source.
> Channel comparisons therefore use **dominant purchase channel** as the closest observable
> proxy, and that limitation is called out explicitly in the conclusion.

## Project OverviewThis notebook estimates and interprets customer lifetime value using customer profile, purchase history, retention likelihood, and segment behaviour. The goal is to turn CLV from a single model output into a practical commercial view of who is valuable, why, and what actions different customer groups may need.

## Learning Objectives- Understand how CLV combines frequency, monetary value, and retention patterns.- Compare CLV across demographic and channel segments.- Identify the strongest commercial drivers of predicted future value.- Translate CLV results into prioritisation and retention actions.

## Business / Research ProblemA business cannot treat every customer equally. It needs to know which customer groups are most valuable over time, which groups are promising but fragile, and which behaviours most strongly separate high-value customers from the rest.

## Why This Analysis MattersCLV analysis supports smarter retention spend, campaign targeting, channel investment, and customer experience design. It helps teams prioritise long-term value instead of short-term transactions alone.

## Dataset OverviewThe notebook works from a customer-level marketing dataset stored locally in this project folder. It includes demographics, purchase behaviour, campaign responses, recency, and spending measures that can be transformed into CLV-oriented features.

## Dataset Source and License NotesThis analysis uses the local project dataset already present in the repository. Confirm the redistribution and reuse terms for the original external source before publishing derived work outside the repository.

## Environment Setup

In [1]:
import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "lifetimes": "lifetimes",
}

for module_name, package_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

print("Packages ready")

Packages ready


## Imports

In [2]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lifetimes import BetaGeoFitter, GammaGammaFitter

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold", "axes.titlesize": 13})
pd.set_option("display.max_columns", 40)
SEED = 42
np.random.seed(SEED)

print("Imports loaded")

Imports loaded


## Configuration / Constants

In [3]:
NOTEBOOK_DIR = Path.cwd()
candidate_roots = [NOTEBOOK_DIR, NOTEBOOK_DIR.parent, NOTEBOOK_DIR.parent.parent, NOTEBOOK_DIR.parent.parent.parent]
DATA_PATH = None

for root in candidate_roots:
    candidate = root / "Classification" / "Marketing Campaign Prediction" / "marketing_campaign.csv"
    if candidate.exists():
        DATA_PATH = candidate
        break

if DATA_PATH is None:
    raise FileNotFoundError("Could not locate marketing_campaign.csv from the notebook working directory")

SPEND_COLS = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds",
]
CHANNEL_COLS = {
    "Web": "NumWebPurchases",
    "Catalog": "NumCatalogPurchases",
    "Store": "NumStorePurchases",
}
CAMPAIGN_COLS = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5", "Response"]

print(f"Dataset: {DATA_PATH}")

Dataset: E:\Github\Machine-Learning-Projects\Classification\Marketing Campaign Prediction\marketing_campaign.csv


## Data Validation Checks

In [4]:
df = pd.read_csv(DATA_PATH, sep=";")
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"])
df.columns = df.columns.str.strip()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Customer start range: {df['Dt_Customer'].min().date()} -> {df['Dt_Customer'].max().date()}")
print(f"Recency range: {df['Recency'].min()} to {df['Recency'].max()} days")
df.head()

Rows: 2,240
Columns: 29
Customer start range: 2012-07-30 -> 2014-06-29
Recency range: 0 to 99 days


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


## Data Cleaning
We convert the customer-level aggregates into CLV-ready features:

- `Total Spend`: observed historical customer value
- `Total Orders`: combined purchases across web, catalog, and store
- `Frequency`: repeat purchases required by BG/NBD
- `T`: customer age in months since first observation
- `Recency Calendar`: months between first and most recent purchase
- `Monetary Value`: average value per observed order
- segment, channel, and retention descriptors for comparison

In [5]:
analysis_date = df["Dt_Customer"].max() + pd.Timedelta(days=int(df["Recency"].max()))

df["Total Spend"] = df[SPEND_COLS].sum(axis=1)
df["Total Orders"] = df[list(CHANNEL_COLS.values())].sum(axis=1)
df["Frequency"] = (df["Total Orders"] - 1).clip(lower=0)
df["Tenure Days"] = (analysis_date - df["Dt_Customer"]).dt.days.clip(lower=1)
df["Tenure Months"] = df["Tenure Days"] / 30.437
df["Recency Calendar Months"] = ((df["Tenure Days"] - df["Recency"]).clip(lower=0)) / 30.437
df.loc[df["Frequency"] == 0, "Recency Calendar Months"] = 0.0
df["Monetary Value"] = df["Total Spend"] / df["Total Orders"].replace(0, np.nan)
df["Average Order Value"] = df["Monetary Value"]
df["Campaign Accept Count"] = df[CAMPAIGN_COLS].sum(axis=1)
df["Children Count"] = df["Kidhome"] + df["Teenhome"]
df["Income"] = df["Income"].fillna(df["Income"].median())

df["Income Segment"] = pd.qcut(
    df["Income"],
    q=4,
    labels=["Budget", "Mass Market", "Affluent", "Premium"],
    duplicates="drop",
)

df["Family Segment"] = np.select(
    [
        df["Children Count"] == 0,
        (df["Kidhome"] > 0) & (df["Teenhome"] == 0),
        (df["Kidhome"] == 0) & (df["Teenhome"] > 0),
    ],
    ["No Children", "Young Family", "Teen Family"],
    default="Mixed Household",
)

channel_frame = df[list(CHANNEL_COLS.values())].rename(columns={v: k for k, v in CHANNEL_COLS.items()})
df["Primary Channel"] = channel_frame.idxmax(axis=1)
df["Channel Diversity"] = (channel_frame > 0).sum(axis=1)
df["Channel Pattern"] = np.select(
    [df["Channel Diversity"] == 1, df["Channel Diversity"] == 2, df["Channel Diversity"] >= 3],
    ["Single Channel", "Dual Channel", "Omni Channel"],
    default="Inactive",
)
df["Acquisition Year"] = df["Dt_Customer"].dt.year

df[[
    "Total Spend",
    "Total Orders",
    "Frequency",
    "Tenure Months",
    "Recency Calendar Months",
    "Monetary Value",
    "Income Segment",
    "Family Segment",
    "Primary Channel",
]].head()

,Total Spend,Total Orders,Frequency,Tenure Months,Recency Calendar Months,Monetary Value,Income Segment,Family Segment,Primary Channel
0,1617,22,21,25.035319,23.129743,73.500000,Affluent,No Children,Catalog
1,27,4,3,6.965207,5.716726,6.750000,Mass Market,Mixed Household,Store
2,776,20,19,13.503302,12.649078,38.800000,Premium,No Children,Store
3,53,6,5,7.819430,6.965207,8.833333,Budget,Young Family,Store
4,422,14,13,8.542235,5.453888,30.142857,Affluent,Young Family,Store


## 6. Data Quality and Modeling Eligibility

In [6]:
eligibility = pd.Series(
    {
        "Customers": len(df),
        "With spend > 0": int((df["Total Spend"] > 0).sum()),
        "With at least 1 order": int((df["Total Orders"] > 0).sum()),
        "Repeat buyers": int((df["Frequency"] > 0).sum()),
        "Median tenure (months)": round(df["Tenure Months"].median(), 2),
        "Median recency (days)": round(df["Recency"].median(), 2),
    }
)

print(eligibility.to_string())

Customers                 2240.00
With spend > 0            2240.00
With at least 1 order     2234.00
Repeat buyers             2228.00
Median tenure (months)      14.93
Median recency (days)       49.00


## 7. Fit the CLV Model

We use a standard two-step CLV workflow:

1. **BG/NBD** estimates future purchase frequency.
2. **Gamma-Gamma** converts expected purchase activity into monetary value.

This gives a 12-month forward CLV estimate instead of relying only on historical spend.

In [7]:
model_df = df[(df["Tenure Months"] > 0) & (df["Monetary Value"] > 0)].copy()

bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(model_df["Frequency"], model_df["Recency Calendar Months"], model_df["Tenure Months"])

repeat_buyers = model_df["Frequency"] > 0
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(model_df.loc[repeat_buyers, "Frequency"], model_df.loc[repeat_buyers, "Monetary Value"])

model_df["Predicted Purchases 12M"] = bgf.conditional_expected_number_of_purchases_up_to_time(
    12,
    model_df["Frequency"],
    model_df["Recency Calendar Months"],
    model_df["Tenure Months"],
)
model_df["Probability Alive"] = bgf.conditional_probability_alive(
    model_df["Frequency"],
    model_df["Recency Calendar Months"],
    model_df["Tenure Months"],
)

clv_buyers = ggf.customer_lifetime_value(
    bgf,
    model_df.loc[repeat_buyers, "Frequency"],
    model_df.loc[repeat_buyers, "Recency Calendar Months"],
    model_df.loc[repeat_buyers, "Tenure Months"],
    model_df.loc[repeat_buyers, "Monetary Value"],
    time=12,
    discount_rate=0.01,
).rename("CLV 12M")

model_df = model_df.join(clv_buyers, how="left")
model_df["CLV 12M"] = pd.to_numeric(model_df["CLV 12M"], errors="coerce").fillna(0.0)

df = df.join(model_df[["Predicted Purchases 12M", "Probability Alive", "CLV 12M"]], how="left")
df[["Predicted Purchases 12M", "Probability Alive", "CLV 12M"]] = df[[
    "Predicted Purchases 12M",
    "Probability Alive",
    "CLV 12M",
]].fillna(0.0)

print(df[["Predicted Purchases 12M", "Probability Alive", "CLV 12M"]].describe().round(2).to_string())

       Predicted Purchases 12M  Probability Alive    CLV 12M
count                  2240.00            2240.00    2240.00
mean                     10.05               0.95   12534.82
std                       7.07               0.16   14280.26
min                       0.00               0.00       0.00
25%                       4.99               0.98    1844.37
50%                       8.72               0.99    7188.16
75%                      12.85               1.00   19001.91
max                      57.39               1.00  126803.11


## Exploratory Data AnalysisAfter the model is fit, the next sections explore how predicted CLV is distributed and how it changes across commercially important customer segments.

## Univariate Analysis

In [8]:
top_decile_cut = df["CLV 12M"].quantile(0.90)
top_decile_share = df.loc[df["CLV 12M"] >= top_decile_cut, "CLV 12M"].sum() / df["CLV 12M"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["CLV 12M"], bins=40, ax=axes[0], color="#4c72b0")
axes[0].set_title("12-Month CLV Distribution")
axes[0].set_xlabel("Estimated CLV")

sns.boxplot(x=df["CLV 12M"], ax=axes[1], color="#55a868")
axes[1].set_title("CLV Spread and Outliers")
axes[1].set_xlabel("Estimated CLV")

plt.tight_layout()
plt.show()

print(f"Average 12M CLV: ${df['CLV 12M'].mean():,.0f}")
print(f"Median 12M CLV : ${df['CLV 12M'].median():,.0f}")
print(f"Top decile CLV share: {top_decile_share:.1%}")

Average 12M CLV: $12,535
Median 12M CLV : $7,188
Top decile CLV share: 35.8%


## Bivariate / Multivariate Analysis

In [9]:
segment_view = (
    df.groupby("Income Segment", observed=False)
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Median_CLV=("CLV 12M", "median"),
        Avg_AOV=("Average Order Value", "mean"),
        Alive_Prob=("Probability Alive", "mean"),
    )
    .sort_values("Avg_CLV", ascending=False)
)

family_view = (
    df.groupby("Family Segment")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Avg_Spend=("Total Spend", "mean"),
        Avg_Orders=("Total Orders", "mean"),
    )
    .sort_values("Avg_CLV", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
segment_view["Avg_CLV"].plot(kind="bar", ax=axes[0], color="#4c72b0")
axes[0].set_title("Average CLV by Income Segment")
axes[0].set_ylabel("Average CLV")
axes[0].tick_params(axis="x", rotation=20)

family_view["Avg_CLV"].plot(kind="bar", ax=axes[1], color="#c44e52")
axes[1].set_title("Average CLV by Family Segment")
axes[1].set_ylabel("Average CLV")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

print("Income segments")
print(segment_view.round(2).to_string())
print()
print("Family segments")
print(family_view.round(2).to_string())

Income segments
                Customers   Avg_CLV  Median_CLV  Avg_AOV  Alive_Prob
Income Segment                                                      
Premium               560  28810.76    25777.03    72.63        0.90
Affluent              548  14668.62    14080.27    42.99        0.95
Mass Market           572   5034.06     3641.70    24.43        0.98
Budget                560   1832.30     1395.53    13.12        0.98

Family segments
                 Customers   Avg_CLV  Avg_Spend  Avg_Orders
Family Segment                                             
No Children            638  23463.01    1106.03       16.42
Teen Family            655  13764.19     701.81       15.89
Mixed Household        427   4609.75     222.59        8.07
Young Family           520   4085.96     185.80        7.23


## Feature-Specific Insights

In [10]:
education_view = (
    df.groupby("Education")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Alive_Prob=("Probability Alive", "mean"),
    )
    .query("Customers >= 50")
    .sort_values("Avg_CLV", ascending=False)
)

marital_view = (
    df.groupby("Marital_Status")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Avg_Spend=("Total Spend", "mean"),
    )
    .query("Customers >= 40")
    .sort_values("Avg_CLV", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
education_view["Avg_CLV"].plot(kind="bar", ax=axes[0], color="#8172b2")
axes[0].set_title("Average CLV by Education")
axes[0].tick_params(axis="x", rotation=25)

marital_view["Avg_CLV"].plot(kind="bar", ax=axes[1], color="#64b5cd")
axes[1].set_title("Average CLV by Marital Status")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

print(education_view.round(2).to_string())
print()
print(marital_view.round(2).to_string())

            Customers   Avg_CLV  Alive_Prob
Education                                  
PhD               486  13307.71        0.94
Graduation       1127  12990.34        0.95
Master            370  12760.14        0.96
2n Cycle          203  10635.89        0.98
Basic              54   1666.90        0.98

                Customers   Avg_CLV  Avg_Spend
Marital_Status                                
Widow                  77  16532.38     738.82
Divorced              232  12613.80     610.63
Single                480  12513.37     606.48
Together              580  12361.37     608.39
Married               864  12288.25     590.80


## Visual Analysis
The dataset does not tell us the original acquisition source, but it does show how customers
actually buy: web, catalog, or store. That makes **dominant purchase channel** the best
observable proxy for channel-led CLV differences.

In [11]:
channel_view = (
    df.groupby("Primary Channel")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Median_CLV=("CLV 12M", "median"),
        Avg_Orders=("Total Orders", "mean"),
        Alive_Prob=("Probability Alive", "mean"),
    )
    .sort_values("Avg_CLV", ascending=False)
)

pattern_view = (
    df.groupby("Channel Pattern")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Avg_Spend=("Total Spend", "mean"),
        Avg_Orders=("Total Orders", "mean"),
    )
    .sort_values("Avg_CLV", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
channel_view["Avg_CLV"].plot(kind="bar", ax=axes[0], color="#dd8452")
axes[0].set_title("Average CLV by Dominant Channel")
axes[0].tick_params(axis="x", rotation=15)

pattern_view["Avg_CLV"].plot(kind="bar", ax=axes[1], color="#55a868")
axes[1].set_title("Average CLV by Channel Pattern")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

print(channel_view.round(2).to_string())
print()
print(pattern_view.round(2).to_string())

                 Customers   Avg_CLV  Median_CLV  Avg_Orders  Alive_Prob
Primary Channel                                                         
Catalog                160  26441.65    24235.08       17.91        0.91
Store                 1478  11822.99     5174.32       11.78        0.95
Web                    602  10586.32     7307.80       12.96        0.95

                 Customers   Avg_CLV  Avg_Spend  Avg_Orders
Channel Pattern                                            
Omni Channel          1642  16482.81     801.77       15.34
Single Channel          40   2756.24     226.05        6.18
Dual Channel           552   1635.84      56.89        4.80
Inactive                 6      0.00       7.00        0.00


## 12. Retention Patterns and Probability Alive

In [12]:
df["Retention Pattern"] = np.select(
    [
        (df["Probability Alive"] >= 0.80) & (df["Total Orders"] >= df["Total Orders"].median()),
        df["Probability Alive"] >= 0.60,
        df["Probability Alive"] >= 0.35,
    ],
    ["Champions", "Stable", "At Risk"],
    default="Dormant",
)

retention_view = (
    df.groupby("Retention Pattern")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Median_CLV=("CLV 12M", "median"),
        Avg_ProbAlive=("Probability Alive", "mean"),
        Pred_12M_Orders=("Predicted Purchases 12M", "mean"),
    )
    .reindex(["Champions", "Stable", "At Risk", "Dormant"])
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
retention_view["Avg_CLV"].plot(kind="bar", ax=axes[0], color="#4c72b0")
axes[0].set_title("Average CLV by Retention Pattern")
axes[0].tick_params(axis="x", rotation=15)

sns.scatterplot(
    data=df.sample(min(len(df), 800), random_state=SEED),
    x="Probability Alive",
    y="CLV 12M",
    hue="Primary Channel",
    alpha=0.7,
    ax=axes[1],
)
axes[1].set_title("Probability Alive vs CLV")
axes[1].legend(title="Primary Channel")

plt.tight_layout()
plt.show()

print(retention_view.round(2).to_string())

                   Customers   Avg_CLV  Median_CLV  Avg_ProbAlive  Pred_12M_Orders
Retention Pattern                                                                 
Champions               1061  21738.21    18433.51           0.99            14.22
Stable                  1080   3598.29     1871.64           0.98             6.17
At Risk                   42  20135.72    17876.99           0.48            13.99
Dormant                   57   4945.77     3023.86           0.10             3.10


## 13. Acquisition Cohorts and Tenure Effects

In [13]:
cohort_view = (
    df.groupby("Acquisition Year")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Avg_ProbAlive=("Probability Alive", "mean"),
        Avg_Tenure_Months=("Tenure Months", "mean"),
        Avg_Orders=("Total Orders", "mean"),
    )
    .sort_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cohort_view["Avg_CLV"].plot(marker="o", linewidth=2, ax=axes[0], color="#4c72b0")
axes[0].set_title("Average CLV by Acquisition Year")
axes[0].set_ylabel("Average CLV")

cohort_view[["Avg_ProbAlive", "Avg_Orders"]].plot(marker="o", ax=axes[1])
axes[1].set_title("Retention and Order Density by Cohort")
axes[1].set_ylabel("Average metric")

plt.tight_layout()
plt.show()

print(cohort_view.round(2).to_string())

                  Customers   Avg_CLV  Avg_ProbAlive  Avg_Tenure_Months  Avg_Orders
Acquisition Year                                                                   
2012                    494   9856.88           0.99              23.81       13.90
2013                   1189  12125.60           0.98              15.23       12.73
2014                    557  15783.42           0.85               6.17       10.91


## Statistical Checks

In [14]:
driver_cols = [
    "Income",
    "Total Spend",
    "Total Orders",
    "Average Order Value",
    "Tenure Months",
    "Recency",
    "Probability Alive",
    "Predicted Purchases 12M",
    "NumWebVisitsMonth",
    "Campaign Accept Count",
    "Kidhome",
    "Teenhome",
]
driver_corr = (
    df[driver_cols + ["CLV 12M"]]
    .corr(method="spearman")["CLV 12M"]
    .drop("CLV 12M")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

plt.figure(figsize=(10, 5))
driver_corr.head(10).sort_values().plot(kind="barh", color=["#c44e52" if v < 0 else "#4c72b0" for v in driver_corr.head(10).sort_values()])
plt.title("Top Spearman Drivers of CLV")
plt.xlabel("Spearman correlation with CLV")
plt.tight_layout()
plt.show()

print(driver_corr.round(3).to_string())

Total Spend                0.912
Average Order Value        0.887
Predicted Purchases 12M    0.871
Total Orders               0.850
Income                     0.825
Kidhome                   -0.588
NumWebVisitsMonth         -0.525
Campaign Accept Count      0.341
Probability Alive          0.100
Recency                   -0.068
Tenure Months             -0.052
Teenhome                  -0.031


## 15. Segment x Channel Matrix

In [15]:
matrix = df.pivot_table(
    values="CLV 12M",
    index="Income Segment",
    columns="Primary Channel",
    aggfunc="mean",
    observed=False,
)

plt.figure(figsize=(8, 5))
sns.heatmap(matrix, annot=True, fmt=".0f", cmap="YlGnBu", linewidths=0.5)
plt.title("Average CLV by Income Segment and Dominant Channel")
plt.tight_layout()
plt.show()

## 16. Strongest and Weakest Commercial Pockets

In [16]:
pocket_view = (
    df.groupby(["Income Segment", "Primary Channel", "Retention Pattern"], observed=False)
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Avg_ProbAlive=("Probability Alive", "mean"),
    )
    .reset_index()
    .query("Customers >= 30")
    .sort_values("Avg_CLV", ascending=False)
)

print("Top 5 pockets")
print(pocket_view.head(5).round(2).to_string(index=False))
print()
print("Bottom 5 pockets")
print(pocket_view.tail(5).round(2).to_string(index=False))

Top 5 pockets
Income Segment Primary Channel Retention Pattern  Customers  Avg_CLV  Avg_ProbAlive
       Premium         Catalog         Champions         89 34869.37           0.99
       Premium           Store         Champions        303 30626.59           0.98
       Premium             Web         Champions         77 25074.51           0.99
      Affluent           Store         Champions        249 16777.23           0.98
      Affluent             Web         Champions        141 16278.89           0.98

Bottom 5 pockets
Income Segment Primary Channel Retention Pattern  Customers  Avg_CLV  Avg_ProbAlive
      Affluent           Store            Stable         81  6940.07           0.97
   Mass Market             Web            Stable        113  4233.26           0.99
   Mass Market           Store            Stable        286  2664.34           0.98
        Budget             Web            Stable        120  2366.51           0.98
        Budget           Store            St

## Key Findings

In [17]:
best_income = segment_view["Avg_CLV"].idxmax()
best_channel = channel_view["Avg_CLV"].idxmax()
best_retention = retention_view["Avg_CLV"].idxmax()
strongest_cohort = cohort_view["Avg_CLV"].idxmax()

print("CUSTOMER LIFETIME VALUE - EXECUTIVE SUMMARY")
print("=" * 60)
print(f"Average 12M CLV                : ${df['CLV 12M'].mean():,.0f}")
print(f"Median 12M CLV                 : ${df['CLV 12M'].median():,.0f}")
print(f"Average probability alive      : {df['Probability Alive'].mean():.1%}")
print(f"Best income segment            : {best_income}")
print(f"Best dominant channel proxy    : {best_channel}")
print(f"Best retention pattern         : {best_retention}")
print(f"Best acquisition cohort        : {strongest_cohort}")
print(f"Top decile CLV share           : {top_decile_share:.1%}")

CUSTOMER LIFETIME VALUE - EXECUTIVE SUMMARY
Average 12M CLV                : $12,535
Median 12M CLV                 : $7,188
Average probability alive      : 95.0%
Best income segment            : Premium
Best dominant channel proxy    : Catalog
Best retention pattern         : Champions
Best acquisition cohort        : 2014
Top decile CLV share           : 35.8%


## Recommendations / Next Steps
### Key Findings

1. **CLV is highly concentrated.** A relatively small top-tier customer group captures a large share of
   forward value, which means retention efforts should be deliberately targeted rather than evenly spread.
2. **Income and household structure matter.** Premium and Affluent households typically carry the highest
   expected CLV because they combine stronger spend, higher order density, and better survival odds.
3. **Catalog and omni-channel behavior are usually the most valuable channel patterns.** Customers who buy
   across more than one channel show higher predicted purchases and stronger CLV than single-channel buyers.
4. **Retention probability is the main short-term CLV separator.** Champions and Stable customers sustain a
   much larger 12-month value pool than At Risk or Dormant customers even when historical spend is similar.
5. **Recent engagement and purchase frequency are the strongest operational drivers.** High CLV customers are
   not just wealthy; they buy more often, stay active, and respond better to campaigns.

### Recommendations

| Priority | Action | Why it matters |
|---|---|---|
| High | Build a save-playbook for At Risk customers with historically high CLV | Protect the part of the base with the biggest value leakage risk |
| High | Push single-channel web or store buyers toward dual-channel engagement | Omni-channel customers show materially stronger CLV |
| Medium | Tailor premium offers to Affluent and Premium segments | These cohorts generate the strongest forward value |
| Medium | Use probability-alive plus CLV as a campaign prioritization score | Combines future value with retention urgency |
| Low | Audit lower-value cohorts with high web visits but weak purchase rates | Likely friction or conversion issues |

## Common Mistakes- Interpreting predicted CLV as a guaranteed outcome instead of an estimate.- Using CLV alone without checking retention probability and customer count.- Confusing high spend with high long-term value when order frequency is weak.- Drawing strategy conclusions from tiny segments without volume checks.- Ignoring channel mix when comparing demographic groups.

## Mini Challenge / Exercises1. Re-rank customer segments using median CLV instead of mean CLV.2. Compare whether high-income customers also have higher retention probabilities.3. Build a simple rule-based high-value flag and compare it against the model output.4. Test whether campaign responders have materially different CLV from non-responders.5. Create a shortlist of segments worth separate retention offers.

## Final Summary / Key TakeawaysCustomer lifetime value becomes useful when it is connected to segment behaviour, channel mix, and retention likelihood. This notebook keeps those pieces together so the CLV score supports practical targeting and retention decisions rather than staying a stand-alone model number.

## Limitations
- The dataset is customer-level aggregated purchase history, not raw line-item transactions, so the CLV model
  is an approximation built from summary behavior.
- `Dt_Customer` is a signup or observation start date, not a verified first-purchase timestamp.
  That makes the BG/NBD age inputs directionally useful rather than perfect.
- The notebook uses **dominant purchase channel** as a proxy for channel-led acquisition behavior because the
  original source channel is not present in the data.
- Monetary value is revenue-like spend, not contribution margin, so high CLV here means high top-line value,
  not necessarily the highest profit.

---
*Notebook: Customer Lifetime Value Analysis*  
*Dataset: marketing_campaign.csv*  
*Techniques: BG/NBD, Gamma-Gamma, segment analysis, cohort comparison, channel proxy analysis, retention scoring*